# 🤖 Notebook 2: NLP Chatbot Development**AI-Driven Insurance Chatbot — Diploma Project**This notebook builds and tests the TF-IDF based chatbot enginethat answers pharmacist questions about insurance policies.---

## 1. Setup

In [ ]:
!pip install scikit-learn pandas -qimport jsonimport reimport numpy as npfrom difflib import SequenceMatcherfrom sklearn.feature_extraction.text import TfidfVectorizerfrom sklearn.metrics.pairwise import cosine_similarityprint('All packages loaded!')

## 2. Load the Knowledge BaseLoad the JSON knowledge base built in Notebook 1.

In [ ]:
# Upload or load the knowledge base# from google.colab import files# uploaded = files.upload()  # Upload insurance_knowledge_base.jsonwith open('insurance_knowledge_base.json', 'r', encoding='utf-8') as f:    kb = json.load(f)print(f'Loaded {len(kb)} companies')

## 3. Arabic Text PreprocessingArabic text needs special normalization for effective matching:- Remove diacritics (tashkeel): فَتْحَة → فتحة- Normalize alef variants: أ إ آ → ا- Normalize ta marbuta: ة → ه- Normalize ya: ى → ي

In [ ]:
def normalize_arabic(text):    """Normalize Arabic text for better matching."""    if not text:        return ''    # Remove diacritics    text = re.sub(r'[\u064B-\u0652\u0670]', '', text)    # Normalize alef variants    text = re.sub(r'[أإآ]', 'ا', text)    # Normalize ta marbuta    text = text.replace('ة', 'ه')    # Normalize ya    text = text.replace('ى', 'ي')    # Lowercase    text = text.lower()    return text# Test normalizationtest_words = ['يونايتد', 'محظورات', 'التأمين', 'الروشتة']for w in test_words:    print(f'  {w} → {normalize_arabic(w)}')

## 4. Build TF-IDF Index**TF-IDF (Term Frequency - Inverse Document Frequency)** weights words by:- **TF**: How often a word appears in a document- **IDF**: How rare a word is across all documentsWords that are frequent in one document but rare overall get high scores.

In [ ]:
# Build documents from the knowledge basedocuments = []chunk_index = []  # Maps row -> (company, category)STOP_WORDS = {'في', 'من', 'على', 'إلى', 'عن', 'مع', 'هذا', 'هذه', 'التي',              'الذي', 'هو', 'هي', 'أن', 'كان', 'كل', 'لم', 'لن', 'يتم',              'يجب', 'لابد', 'و', 'أو', 'لا', 'ما', 'فى', 'بعد', 'قبل'}for comp_key, comp_data in kb.items():    for cat, policy in comp_data.get('policies', {}).items():        chunk = ' '.join([            comp_data.get('company_name', ''),            cat,            policy.get('details', ''),            policy.get('notes', ''),        ])        documents.append(normalize_arabic(chunk))        chunk_index.append((comp_key, cat))print(f'Total documents (chunks): {len(documents)}')print(f'Sample document: {documents[0][:200]}...')

In [ ]:
# Build TF-IDF matrixvectorizer = TfidfVectorizer(    max_features=5000,    ngram_range=(1, 2),  # Unigrams + bigrams    stop_words=list(STOP_WORDS),)tfidf_matrix = vectorizer.fit_transform(documents)print(f'TF-IDF matrix shape: {tfidf_matrix.shape}')print(f'Vocabulary size: {len(vectorizer.vocabulary_)}')# Show top featuresfeature_names = vectorizer.get_feature_names_out()print(f'\nSample features: {list(feature_names[:20])}')

## 5. Test the Search EngineLet's test with real pharmacist questions:

In [ ]:
def search(query, top_k=3):    """Search the knowledge base using TF-IDF cosine similarity."""    query_norm = normalize_arabic(query)    query_vec = vectorizer.transform([query_norm])    similarities = cosine_similarity(query_vec, tfidf_matrix).flatten()    top_indices = similarities.argsort()[-top_k:][::-1]    results = []    for idx in top_indices:        if similarities[idx] < 0.01:            continue        comp_key, cat = chunk_index[idx]        comp = kb[comp_key]        policy = comp['policies'][cat]        results.append({            'company': comp.get('company_name', ''),            'category': cat,            'details': policy.get('details', '')[:200],            'score': float(similarities[idx]),        })    return results# Test queriestest_queries = [    'ما هي محظورات يونايتد',    'أقصى مدة صرف لشركة ويبكو',    'رقم تليفون موافقات دريم مشرق',    'هل يشترط ختم لشركة جلوبميد',    'What is the copay for ALICO',]for q in test_queries:    print(f'\n🔍 Query: {q}')    results = search(q)    for r in results:        print(f'  [{r["score"]:.3f}] {r["company"]} — {r["category"]}')        print(f'         {r["details"][:100]}...')

## 6. Evaluate Retrieval AccuracyTest with known question-answer pairs to measure precision:

In [ ]:
# Ground truth test casestest_cases = [    {'query': 'محظورات يونايتد', 'expected_company': 'يونايتد-united', 'expected_cat': 'المحظورات'},    {'query': 'أقصى مدة صرف ويبكو', 'expected_company': 'ويبكو-wepco', 'expected_cat': 'أقصى مدة للصرف'},    {'query': 'تواصل موافقات دريم مشرق', 'expected_company': 'شركة دريم مشرق للأغذية-dream mashreq', 'expected_cat': 'التواصل للموافقات'},    {'query': 'ختم جلوبميد', 'expected_company': 'GLOBEMED-جلوبميد', 'expected_cat': 'الختم / إمضاء العميل'},]correct = 0for tc in test_cases:    results = search(tc['query'], top_k=1)    if results:        top = results[0]        match = (tc['expected_company'] in top['company'] or                 top['company'] in tc['expected_company'])        cat_match = tc['expected_cat'] == top['category']        if match and cat_match:            correct += 1            print(f'  ✅ {tc["query"]} → {top["company"]} / {top["category"]}')        else:            print(f'  ❌ {tc["query"]} → Expected: {tc["expected_company"]} / Got: {top["company"]}')    else:        print(f'  ❌ {tc["query"]} → No results')precision = correct / len(test_cases)print(f'\nPrecision@1: {precision:.0%} ({correct}/{len(test_cases)})')

## 7. Summary| Component | Details ||-----------|--------|| Search Method | TF-IDF + Cosine Similarity || Arabic NLP | Diacritics removal, alef/ya normalization || Vocabulary Size | ~5000 features || N-gram Range | Unigrams + Bigrams || Matching Strategy | Direct + Fuzzy + TF-IDF fallback |**Next:** Notebook 3 will evaluate the complete system.